# Adenylate Kinase trajectory analysis

This notebook analyses the stored coarse-grained trajectory using RMSD/Q measures and domain center-of-mass coordinates, then constructs the free-energy landscape along the selected collective coordinates.


## Load the stored trajectory


In [1]:
import numpy as np
import matplotlib.pyplot as plt

import mdtraj as md
from openmm.app.topology import Topology
from openmm.app.simulation import Simulation
from openmm.app.dcdreporter import DCDReporter
from openmm.app.statedatareporter import StateDataReporter
import openmm.unit as u
import openmm as mm
import openmm.openmm as openmm
import pyemma.coordinates as coor
import pyemma

## 1) Domains

Now we want to distinguish between the different domains: 

Residues 30–59 to the NMP domain
Residues 122–159 to the LID domain 
Residues 1–29, 60–121 and 160–214 to the CORE domain

Residues 113–122 make up the distal LID-CORE hinge
Residues 29–31 the distal NMP-CORE hinge
Residues 160–169 the proximal LID-CORE hinge region 
Residues 55–58 the proximal NMPCOREhinge.


In [4]:
NMP_indices = (30, 59)
LID_indices = (122, 159)
CORE_indices = list(range(1, 30)) + list(range(60, 122)) + list(range(160, 214))

traj_AKE4 = md.load('traj_AKE4.xtc', top='pdb_file_4AKE_Calpha.pdb')

NMP_residues_traj_AKE4 = traj_AKE4.atom_slice(range(NMP_indices[0]-1, NMP_indices[1]))
NMP_residues_traj_AKE4.save_xtc('traj_AKE4_NMP.xtc')

LID_residues_traj_AKE4 = traj_AKE4.atom_slice(range(LID_indices[0]-1, LID_indices[1]))
LID_residues_traj_AKE4.save_xtc('traj_AKE4_LID.xtc')

CORE_residues_traj_AKE4 = traj_AKE4.atom_slice(CORE_indices)
CORE_residues_traj_AKE4.save_xtc('traj_AKE4_CORE.xtc')


In [5]:
# We now calculate the center of mass of each domain for every step in the trajectory

center_of_mass_array_NMP = md.compute_center_of_mass(NMP_residues_traj_AKE4)
center_of_mass_array_LID = md.compute_center_of_mass(LID_residues_traj_AKE4)
center_of_mass_array_CORE = md.compute_center_of_mass(CORE_residues_traj_AKE4)



In [ ]:
distance_CORE_NMP = np.linalg.norm(
    center_of_mass_array_NMP - center_of_mass_array_CORE, axis=1
) * 10.0  # MDTraj coordinates are in nm; convert to Å
distance_CORE_LID = np.linalg.norm(
    center_of_mass_array_LID - center_of_mass_array_CORE, axis=1
) * 10.0

plt.scatter(distance_CORE_NMP, distance_CORE_LID)

plt.title('Center of mass (COM) distances between domains')
plt.xlabel('Distance between COM of CORE and NMP domains (Å)')
plt.ylabel('Distance between COM of CORE and LID domains (Å)')

# Show the plot
plt.show()

In [ ]:

pyemma.plots.plot_free_energy(distance_CORE_LID, distance_CORE_NMP)

plt.xlabel('Distance between COM of CORE and LID domains (Å)')
plt.ylabel('Distance between COM of CORE and NMP domains (Å)')
plt.tight_layout()
plt.savefig('../results/free_energy_landscape.png', dpi=300, bbox_inches='tight')
plt.show()
